# Zerobus Ingest

## Data path — no customer-managed queue

```
Producer                        Databricks                   Consumer    
+----------+          +---------------------------+          +----------+
| App w/   |  append  | Zerobus     Unity Catalog | full DML | AI       |
| Zerobus  |--------->| Endpoint -> Delta Table   |<---------| BI       |
| SDK      |          |                           |          | Agents   |
+----------+          +---------------------------+          +----------+
```

*No Kafka / Event Hubs / Kinesis queue you operate between the app and the managed Zerobus endpoint.*

## Setup 

- Auto-configure Zerobus endpoint 
- Auto-configure Service Principal (admin priv required to create SP if not present)
- Catalog, Schema, Table permissions

## Show performance characteristics
  - **~250 ms/row** for single-row inserts
  - **greater than ~250 ms+** for whole-batch insert
  - **~5 s** until all rows are visible

## Show system tables
  - system.lakeflow.zerobus_ingest (rows and bytes)
  - system.lakeflow.zerobus_stream (connections)
---

This notebook follows **[Use the Zerobus Ingest connector](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest?language=Python%20SDK)** (Python SDK / JSON example).

### More links
- [Zerobus overview](https://docs.databricks.com/aws/en/ingestion/zerobus-overview)
- [Python SDK repository](https://github.com/databricks/zerobus-sdk-py)
- [Get workspace URL and Zerobus endpoint](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#get-your-workspace-url-and-zerobus-ingest-endpoint)


### Step 1: Prepare the environment

In [ ]:
%pip install --quiet databricks-zerobus-ingest-sdk

### Step 2: Configuration

### Step 2.a: Workspace and Zerobus endpoint (auto)

**Region:** `SELECT current_metastore()` — parse `cloud:region:…`. If that fails, set `DATABRICKS_REGION_OVERRIDE`.



In [ ]:
import os
from urllib.parse import urlparse
import re
from databricks.sdk import WorkspaceClient

DATABRICKS_REGION_OVERRIDE = ""


def _region_from_zone_or_region_for_zerobus(s: str) -> str:
    """Normalize override values that look like zones (e.g. us-west-2a) to a Zerobus region label."""
    s = (s or "").strip()
    if not s:
        return ""
    if re.fullmatch(r".*-[a-z]$", s):
        return s.rsplit("-", 1)[0]
    if len(s) >= 2 and s[-1].isalpha() and s[-2].isdigit():
        return s[:-1]
    return s


def _region_from_current_metastore() -> str | None:
    try:
        _rows = spark.sql("SELECT current_metastore()").collect()
        raw = str(_rows[0][0] if _rows else "").strip()
    except Exception as ex:
        print(f"current_metastore() skipped ({type(ex).__name__}: {ex})")
        return None
    if not raw:
        return None
    m = re.match(r"^[a-z]+:([^:]+):", raw, flags=re.IGNORECASE)
    if not m:
        return None
    rgn = m.group(1).strip()
    return rgn or None


def _domain_for_host(h: str) -> str:
    if h.endswith(".azuredatabricks.net"):
        return "azuredatabricks.net"
    if h.endswith(".cloud.databricks.com"):
        return "cloud.databricks.com"
    if "gcp.databricks.com" in h:
        return "gcp.databricks.com"
    raise ValueError(
        f"Unsupported Databricks workspace hostname {h!r}. "
        "Expected *.azuredatabricks.net, *.cloud.databricks.com, or *.gcp.databricks.com."
    )


# Connect sets metadata-service auth; clear so WorkspaceClient uses ~/.databrickscfg locally.
if (os.environ.get("DATABRICKS_AUTH_TYPE") or "").strip().lower() == "metadata-service":
    os.environ.pop("DATABRICKS_AUTH_TYPE", None)
    os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
    print(
        "Cleared metadata-service auth env vars for WorkspaceClient() "
        "(Connect-only; SDK will use your Databricks CLI / profile auth)."
    )

_w = WorkspaceClient()
DATABRICKS_WORKSPACE_URL = _w.config.host.rstrip("/")
_host = (urlparse(DATABRICKS_WORKSPACE_URL).hostname or "").lower()
if not _host:
    raise ValueError(f"Could not parse workspace hostname from {DATABRICKS_WORKSPACE_URL!r}.")

_m_gcp_ws = re.fullmatch(r"([0-9]+)\.[0-9]+\.gcp\.databricks\.com", _host)
if _m_gcp_ws:
    DATABRICKS_WORKSPACE_ID = _m_gcp_ws.group(1)
else:
    try:
        DATABRICKS_WORKSPACE_ID = str(_w.get_workspace_id())
    except Exception as ex:
        raise RuntimeError(
            "Could not determine DATABRICKS_WORKSPACE_ID (SDK get_workspace_id failed). "
            "For GCP, use config.host https://<workspace_id>.<shard>.gcp.databricks.com. "
            f"Underlying error: {ex!r}"
        ) from ex

_domain = _domain_for_host(_host)
_region_ov = str(DATABRICKS_REGION_OVERRIDE).strip()

if _region_ov:
    DATABRICKS_REGION = _region_from_zone_or_region_for_zerobus(_region_ov)
    if not DATABRICKS_REGION:
        raise RuntimeError(f"DATABRICKS_REGION_OVERRIDE produced empty region from {_region_ov!r}.")
    print(
        f"Using DATABRICKS_REGION_OVERRIDE={_region_ov!r} -> DATABRICKS_REGION={DATABRICKS_REGION!r} "
        "(skipped metastore)."
    )
else:
    DATABRICKS_REGION = _region_from_current_metastore()
    if not DATABRICKS_REGION:
        raise RuntimeError(
            "Could not derive region from current_metastore(). Set DATABRICKS_REGION_OVERRIDE."
        )

ZEROBUS_INGEST_URL = f"https://{DATABRICKS_WORKSPACE_ID}.zerobus.{DATABRICKS_REGION}.{_domain}"
SERVER_ENDPOINT = ZEROBUS_INGEST_URL
print(f"{DATABRICKS_WORKSPACE_ID=}\n{DATABRICKS_WORKSPACE_URL=}\n{DATABRICKS_REGION=}\n{ZEROBUS_INGEST_URL=}\n{SERVER_ENDPOINT=}")


### Step 2.b: Service principal + secret

- Parse **`SP_NAME`**: `scope--secretKey--jsonField`, or `prefix--scope--secretKey--jsonField`, or a **short** `prefix` with optional `--` tail. Omitted tail segments default to scope `lfczerobusdemo`, secret key `lfczerobusdemo`, then oauth json field `ZEROBUS_OAUTH_SECRET`. A 3-segment name is treated as legacy `scope--key--field` when the third segment looks like an UPPER_SNAKE json key; otherwise as `prefix--scope--key` with the default field.
- **Create the secret scope** and **secret key** if missing (empty JSON `{}` on first create).
- **Create the workspace service principal** if needed (`display_name` equals **`SP_NAME`**).
- **Write initial JSON** (`ZEROBUS_SERVICE_PRINCIPAL_*`, `ZEROBUS_APP_ID`, and the OAuth field key — empty until 2.c mints).
- Does **not** read or mint the OAuth **client secret** (that is **Step 2.c**).

Reference: `notebooks/zerobus_service_principal.ipynb`.


In [ ]:
# Step 2.b: Secret scope/key + service principal + initial JSON (no OAuth client secret mint here).

# Workspace-specific prefix; optional tail: --<scope>--<secretKey>--<oauthJsonField>
SP_NAME = "lfcdemo_zerobus"

_DEFAULT_SP_SECRET_SCOPE = "lfczerobusdemo"
_DEFAULT_SP_SECRET_KEY = "lfczerobusdemo"
_DEFAULT_SP_OAUTH_JSON_FIELD = "ZEROBUS_OAUTH_SECRET"

import json
import re

from databricks.sdk.errors import NotFound, ResourceAlreadyExists, ResourceDoesNotExist


def _looks_like_oauth_json_key(s: str) -> bool:
    """Legacy 3-part refs use an UPPER_SNAKE json field as the third segment."""
    return bool(re.fullmatch(r"[A-Z][A-Z0-9_]*", s))


def _parse_secret_ref(ref: str):
    """Parse scope--secretKey--jsonField or prefix--scope--secretKey--jsonField.

    With a leading workspace prefix, trailing segments may be omitted: missing scope,
    secret key, and oauth json field default to _DEFAULT_SP_SECRET_SCOPE,
    _DEFAULT_SP_SECRET_KEY, and _DEFAULT_SP_OAUTH_JSON_FIELD respectively.

    A 3-segment value is either that legacy form (third segment looks like a json field
    name) or prefix--scope--secretKey with the default oauth json field.
    """
    parts = [p for p in str(ref).strip().split("--") if p != ""]
    if not parts:
        raise ValueError("SP_NAME is empty or only contains '--' separators.")
    if len(parts) > 4:
        raise ValueError(
            "Secret locator: at most 4 '--' segments "
            "(prefix--scope--secretKey--jsonField, or scope--secretKey--jsonField); "
            f"got {ref!r}"
        )
    if len(parts) == 1:
        return (
            _DEFAULT_SP_SECRET_SCOPE,
            _DEFAULT_SP_SECRET_KEY,
            _DEFAULT_SP_OAUTH_JSON_FIELD,
        )
    if len(parts) == 2:
        return parts[1], _DEFAULT_SP_SECRET_KEY, _DEFAULT_SP_OAUTH_JSON_FIELD
    if len(parts) == 3:
        if _looks_like_oauth_json_key(parts[2]):
            return parts[0], parts[1], parts[2]
        return parts[1], parts[2], _DEFAULT_SP_OAUTH_JSON_FIELD
    return parts[1], parts[2], parts[3]


if not str(SP_NAME).strip():
    raise ValueError("SP_NAME is empty.")

_SECRET_SCOPE, _SECRET_KEY, _oauth_field = _parse_secret_ref(SP_NAME)
_config_keys = [
    "ZEROBUS_SERVICE_PRINCIPAL_NAME",
    "ZEROBUS_SERVICE_PRINCIPAL_ID",
    "ZEROBUS_APP_ID",
    _oauth_field,
]


def _put_secret_json(blob: dict) -> None:
    """Write the full JSON value for this scope/key (shared with Step 2.c)."""
    _w.secrets.put_secret(
        scope=_SECRET_SCOPE,
        key=_SECRET_KEY,
        string_value=json.dumps(blob, indent=2),
    )


def _ensure_secret_scope_and_key() -> None:
    try:
        _w.secrets.create_scope(_SECRET_SCOPE)
        print(f"Created secret scope {_SECRET_SCOPE!r}")
    except ResourceAlreadyExists:
        pass
    try:
        _w.secrets.get_secret(_SECRET_SCOPE, _SECRET_KEY)
    except ResourceDoesNotExist:
        _w.secrets.put_secret(scope=_SECRET_SCOPE, key=_SECRET_KEY, string_value="{}")
        print(f"Created empty secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r}")


def _load_saved() -> dict:
    try:
        raw = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY)
        print(f"Loaded config from secret scope={_SECRET_SCOPE!r} key={_SECRET_KEY!r}")
        return json.loads(raw)
    except Exception:
        pass
    _ensure_secret_scope_and_key()
    return {}


def _save_config(updates: dict) -> None:
    try:
        current = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
    except ResourceDoesNotExist:
        current = {}
    current.update(updates)
    _put_secret_json(current)
    print(f"Saved {list(updates.keys())} to secret {_SECRET_SCOPE}/{_SECRET_KEY}")


_saved = _load_saved()
_config = {k: _saved.get(k, "") for k in _config_keys}
_config_original = {k: _saved.get(k, "") for k in _config_keys}

_sp_state = {"replaced_stale_sp": False}


def _ensure_sp() -> None:
    sp_id = _config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")

    def _create_sp():
        _name = str(SP_NAME).strip()
        _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] = _name
        _sp = _w.service_principals.create(display_name=_name)
        _config["ZEROBUS_SERVICE_PRINCIPAL_ID"] = str(_sp.id)
        _config["ZEROBUS_APP_ID"] = str(_sp.application_id)
        print(f"Created SP {_name!r} id={_sp.id} APP_ID={_config['ZEROBUS_APP_ID']}")

    if not str(sp_id).strip():
        print("ZEROBUS_SERVICE_PRINCIPAL_ID not set — creating service principal")
        _create_sp()
        return

    try:
        _sp = _w.service_principals.get(sp_id)
        print(f"SP exists: {_sp.display_name!r} sp_id={sp_id}")
        api_app = str(_sp.application_id)
        stored_app = str(_config.get("ZEROBUS_APP_ID", "")).strip()
        if stored_app and stored_app != api_app:
            raise RuntimeError(
                f"Secret ZEROBUS_APP_ID={stored_app!r} does not match workspace SP application_id={api_app!r} "
                f"for sp_id={sp_id!r}. Fix the secret JSON or the service principal."
            )
        if not stored_app:
            _config["ZEROBUS_APP_ID"] = api_app
            print(f"Backfilled ZEROBUS_APP_ID={_config['ZEROBUS_APP_ID']}")
    except NotFound:
        print(f"SP id={sp_id!r} not found — creating a replacement")
        _sp_state["replaced_stale_sp"] = True
        _create_sp()


_ensure_sp()

_o_sp = str(_config_original.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip()
_o_app = str(_config_original.get("ZEROBUS_APP_ID", "")).strip()
_n_sp = str(_config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip()
_n_app = str(_config.get("ZEROBUS_APP_ID", "")).strip()

# Once both ids were stored, they must not drift unless we replaced a deleted SP (stale id in the secret).
if _o_sp and _o_app and not _sp_state["replaced_stale_sp"]:
    if _n_sp != _o_sp or _n_app != _o_app:
        raise RuntimeError(
            "Refusing to persist: ZEROBUS_SERVICE_PRINCIPAL_ID or ZEROBUS_APP_ID would change. "
            f"stored sp_id={_o_sp!r} app_id={_o_app!r}; after Step 2.b got sp_id={_n_sp!r} app_id={_n_app!r}. "
            "If you rotated the service principal, update or clear the secret JSON manually."
        )

# Persist when we first record SP metadata, backfill a missing APP_ID, replace a deleted SP, or align name/oauth keys.
# Steady re-runs with the same JSON produce no updates (not an error—just no write).
_updates = {k: _config[k] for k in _config_keys if _config.get(k) != _config_original.get(k)}
if _updates:
    _save_config(_updates)

if not str(_config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip():
    raise RuntimeError(
        "Step 2.b: ZEROBUS_SERVICE_PRINCIPAL_ID is still empty after bootstrap; cannot continue."
    )
if not str(_config.get("ZEROBUS_APP_ID", "")).strip():
    raise RuntimeError("Step 2.b: ZEROBUS_APP_ID is empty after bootstrap; cannot continue.")


### Step 2.c: OAuth client secret

- **Read** the JSON from the same secret **Step 2.b** configured (`_SECRET_SCOPE` / `_SECRET_KEY` / `_oauth_field`).
- **Validate** the stored client secret with the workspace **OIDC** token endpoint.
- If **missing or invalid**, **mint** a new OAuth client secret, **write** the updated JSON back to the secret, then set **`CLIENT_ID`** / **`CLIENT_SECRET`** for later cells.


In [ ]:
# Step 2.c: Read client secret from the Step 2.b secret; OIDC-validate; mint + persist if invalid.
# Requires Step 2.b (defines _SECRET_SCOPE, _SECRET_KEY, _oauth_field, _put_secret_json) and Step 2.a (_w).

import json
import urllib.error
import urllib.request
from urllib.parse import urlencode


def _credentials_valid(client_id: str, client_secret: str, workspace_url: str) -> bool:
    """True if client_id + client_secret work against the workspace OIDC token endpoint."""
    if not client_id or not client_secret:
        return False
    try:
        body = urlencode(
            {
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "all-apis",
            }
        ).encode()
        req = urllib.request.Request(
            f"{workspace_url.rstrip('/')}/oidc/v1/token",
            data=body,
            method="POST",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        with urllib.request.urlopen(req, timeout=15) as resp:
            return 200 <= getattr(resp, "status", 200) < 300
    except urllib.error.HTTPError as ex:
        try:
            msg = ex.read().decode(errors="replace")
        except Exception:
            msg = str(ex)
        print(f"OIDC check: HTTP {ex.code} {msg[:500]}")
        return False
    except Exception as ex:
        print(f"OIDC check: {ex}")
        return False


for _need in ("_SECRET_SCOPE", "_SECRET_KEY", "_oauth_field", "_put_secret_json"):
    if _need not in globals():
        raise RuntimeError(
            f"Step 2.c requires Step 2.b first (missing {_need}). Re-run Step 2.b after restarting the kernel if needed."
        )

_blob = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
if "ZEROBUS_APP_ID" not in _blob:
    raise KeyError(
        "Secret JSON must include ZEROBUS_APP_ID (OAuth application id). Run Step 2.b to create the service principal."
    )
if _oauth_field not in _blob:
    _blob[_oauth_field] = ""

client_id = str(_blob["ZEROBUS_APP_ID"]).strip()
client_secret = str(_blob.get(_oauth_field, "") or "").strip()
_ws_url = _w.config.host.rstrip("/")

if _credentials_valid(client_id, client_secret, _ws_url):
    print('CLIENT_SECRET="***" OAuth client secret is valid (OIDC client_credentials).')
else:
    print("OAuth client secret missing or invalid — minting a new client secret…")
    sp_id = str(_blob.get("ZEROBUS_SERVICE_PRINCIPAL_ID") or "").strip()
    if not sp_id:
        raise RuntimeError(
            "Cannot mint OAuth client secret: ZEROBUS_SERVICE_PRINCIPAL_ID missing in secret JSON. Run Step 2.b."
        )
    _secret_obj = None
    try:
        _secret_obj = _w.service_principal_secrets_proxy.create(service_principal_id=sp_id)
    except AttributeError:
        try:
            from databricks.sdk import ServicePrincipalSecretsAPI

            _secret_obj = ServicePrincipalSecretsAPI(_w.api_client).create(service_principal_id=sp_id)
        except Exception:
            _resp = _w.api_client.do(
                "POST",
                f"/api/2.0/accounts/servicePrincipals/{sp_id}/credentials/secrets",
            )
            _blob[_oauth_field] = _resp["secret"]

    if _secret_obj is not None:
        _blob[_oauth_field] = _secret_obj.secret

    client_secret = str(_blob.get(_oauth_field, "") or "").strip()
    if not _credentials_valid(client_id, client_secret, _ws_url):
        raise RuntimeError(
            "New client secret failed OIDC check; verify workspace URL and SP permissions."
        )
    _put_secret_json(_blob)
    print(
        f"Saved new OAuth client secret to secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r} (field {_oauth_field!r})."
    )

CLIENT_ID = client_id
CLIENT_SECRET = client_secret
if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Step 2.c: CLIENT_ID or CLIENT_SECRET is empty after load/mint.")

print(f"{CLIENT_ID=}")


### Step 2.d: Catalog, schema, and table

Set `CATALOG`, `SCHEMA`, and `TABLE` (doc-style example: `main.<schema>.airquality_grpc_sync`). Blank **`CATALOG`** uses the Spark session default, but if that is **`hive_metastore`** (legacy / default storage), the notebook picks another catalog from `SHOW CATALOGS` — **Zerobus only supports [Unity Catalog managed Delta tables](https://docs.databricks.com/gcp/en/ingestion/zerobus-limits)** (not default storage; error **4024** if wrong). Blank **`SCHEMA`** defaults to a UC-safe name from the current Databricks user (e.g. `robert.lee@…` → `robert_lee`). Builds `TABLE_NAME` for later steps. Uses `spark`, `_w`, and `re` from Step 2.a; **`CLIENT_ID`** / **`CLIENT_SECRET`** come from **Step 2.c**.


In [ ]:
# Table (Step 2.d). Default table segment: airquality_grpc_sync.
# Zerobus requires a UC *managed* Delta table — not hive_metastore / default storage (gRPC 4024).
CATALOG = ""  # blank → Spark default, avoiding hive_metastore when possible; or set explicitly, e.g. "main"
SCHEMA = ""  # blank → UC schema from Databricks username (e.g. robert.lee@… → robert_lee)
TABLE = "airquality_grpc_sync"

_ZB_EXCLUDED_CATALOGS = frozenset({"", "hive_metastore", "spark_catalog"})


def _catalog_name_from_show_row(row) -> str:
    if hasattr(row, "catalogName"):
        return str(row.catalogName)
    d = row.asDict(recursive=True)
    for k in ("catalogName", "catalog", "namespace"):
        if k in d and d[k] is not None:
            return str(d[k])
    return str(row[0])


def _default_catalog_for_zerobus() -> str:
    cur = spark.sql("SELECT current_catalog()").collect()[0][0]
    if str(cur).strip().lower() not in _ZB_EXCLUDED_CATALOGS:
        return str(cur)
    names = []
    for row in spark.sql("SHOW CATALOGS").collect():
        n = _catalog_name_from_show_row(row).strip()
        if n.lower() not in _ZB_EXCLUDED_CATALOGS:
            names.append(n)
    if not names:
        raise RuntimeError(
            "Zerobus Ingest needs a Unity Catalog catalog (managed storage). current_catalog() is "
            f"{cur!r} and no other catalogs are listed. Set CATALOG explicitly in this cell."
        )
    _main = next((x for x in names if x.lower() == "main"), None)
    if _main is not None:
        print(f"current_catalog was {cur!r}; using {_main!r} for UC-managed storage (Zerobus).")
        return _main
    names.sort(key=str.lower)
    print(f"current_catalog was {cur!r}; using {names[0]!r} for UC-managed storage (Zerobus).")
    return names[0]


if not str(CATALOG).strip():
    CATALOG = _default_catalog_for_zerobus()
    print(f"CATALOG default: {CATALOG!r}")

if not str(SCHEMA).strip():
    SCHEMA = re.sub(r"[^a-z0-9]", "_", _w.current_user.me().user_name.split("@")[0].lower())
    print(f"SCHEMA default (from current user): {SCHEMA!r}")

TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"


### Step 3: Create your target table

Schema matches the [docs JSON example](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest?language=Python%20SDK). The next code cell creates the schema and a **managed** Delta table (`USING DELTA`) with fully qualified names, then grants the service principal **USE CATALOG**, **USE SCHEMA**, **SELECT**, and **MODIFY** in that order ([docs](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#create-a-service-principal-and-grant-permissions)). If you still see Zerobus **Unsupported table kind / default storage (4024)**, the table may have been created earlier under `hive_metastore` — `DROP TABLE` it and re-run this step after setting **`CATALOG`** in Step 2.d to a UC catalog.


In [ ]:
# DDL then SP grants (grants require schema + table to exist).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
  device_name STRING,
  temp        INT,
  humidity    BIGINT
) USING DELTA
""")
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `" + CLIENT_ID + "`;").collect()
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `" + CLIENT_ID + "`;").collect()
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {TABLE_NAME} TO `" + CLIENT_ID + "`;").collect()


### Step 4: Doc example — JSON sync client

Adapted from **JSON example** in [Use the Zerobus Ingest connector](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest?language=Python%20SDK). `SERVER_ENDPOINT` and `DATABRICKS_WORKSPACE_URL` come from Step 2.a; `CLIENT_ID` / `CLIENT_SECRET` from Step 2.c.

Set `_n` in the **first Step 4 code cell** (setup), then run **4a**, then **4b**, then **4c** in order. The stream stays open between 4a and 4b; **4b** closes the stream.


In [ ]:
import json
import logging
import time

from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

logging.basicConfig(level=logging.INFO)

_n = 1000  # docs use 1000; lower for a quick test

# Baseline (small demo table: COUNT + MIN/MAX are cheap; no extra index needed)
_row_before = spark.sql(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}").collect()[0]["c"]
_bounds_before = spark.sql(
    f"SELECT MIN(device_name) AS lo, MAX(device_name) AS hi FROM {TABLE_NAME}"
).collect()[0]
print(
    f"Before ingest: count={_row_before} min(device_name)={_bounds_before['lo']!r} max(device_name)={_bounds_before['hi']!r}"
)

sdk = ZerobusSdk(SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL)

table_properties = TableProperties(TABLE_NAME)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties, options)

_singles = min(10, _n)


def _json_payload_bytes(obj: dict) -> int:
    # UTF-8 length of JSON (compact); aligns with byte-oriented ingest / billing discussion
    return len(json.dumps(obj, separators=(",", ":")).encode("utf-8"))


#### 4a. Single-record ingest (first up to 10 rows)

Each row uses `ingest_record_offset` plus `wait_for_offset`. If `_n` is below 10, only `_n` rows are ingested here.


In [ ]:
_row_ack_seconds = []
_singles_wall_s = 0.0
_bytes_4a = 0

t_4a0 = time.perf_counter()
for i in range(_singles):
    record_dict = {
        "device_name": f"sensor-{i}",
        "temp": 20 + i % 15,
        "humidity": 50 + i % 40,
    }
    _bytes_4a += _json_payload_bytes(record_dict)
    t_row = time.perf_counter()
    offset = stream.ingest_record_offset(record_dict)
    stream.wait_for_offset(offset)
    _row_ack_seconds.append(time.perf_counter() - t_row)
_singles_wall_s = time.perf_counter() - t_4a0


#### 4b. Batch ingest (remaining rows)

If `_n > 10`, the rest are sent with `ingest_records_offset` and a single `wait_for_offset` on the final offset. If `_n <= 10`, this step only closes the stream.


In [ ]:
_batch_lo = _singles
_batch_ack_s = None
_bytes_4b = 0
try:
    if _batch_lo < _n:
        batch_records = [
            {
                "device_name": f"sensor-{i}",
                "temp": 20 + i % 15,
                "humidity": 50 + i % 40,
            }
            for i in range(_batch_lo, _n)
        ]
        _bytes_4b = sum(_json_payload_bytes(r) for r in batch_records)
        t_4b0 = time.perf_counter()
        offset = stream.ingest_records_offset(batch_records)
        stream.wait_for_offset(offset)
        _batch_ack_s = time.perf_counter() - t_4b0
finally:
    stream.close()

_batch_n = max(0, _n - _singles)
_ingest_4a4b_s = _singles_wall_s + (_batch_ack_s or 0.0)
_bytes_payload_total = _bytes_4a + _bytes_4b


#### 4c. Lakehouse visibility and Key metrics

Poll `COUNT(*)` until ingested rows are visible in UC, then print **Key metrics** (elapsed time to visibility, single-row and batch timing, JSON payload bytes, and the recap block).


In [ ]:
# Lakehouse visibility: table can lag a few seconds after SDK acks complete
from statistics import mean, median

_target_count = _row_before + _n
_poll_deadline = time.perf_counter() + 120.0
_poll_t0 = time.perf_counter()
_row_visible = _row_before
while time.perf_counter() < _poll_deadline:
    _row_visible = spark.sql(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}").collect()[0]["c"]
    if _row_visible >= _target_count:
        break
    time.sleep(0.15)

_visibility_s = time.perf_counter() - _poll_t0

print("Key metrics:")
print(f"elapsed time to see all rows in UC table={_visibility_s:.3f}")
if _singles:
    print(
        f"single row insert response time ~{(_singles_wall_s / _singles) * 1000:.1f} ms/row amortized"
    )
if _batch_ack_s is not None and _batch_n:
    _key_ms_4b = (_batch_ack_s / _batch_n) * 1000
    print(
        f"batch row insert response time {_batch_ack_s:.3f}s (~{_key_ms_4b:.3f} ms/row amortized "
        f"with {_batch_n:,} rows in batch)"
    )
if _bytes_payload_total:
    print(
        f"JSON payload (UTF-8, compact): {_bytes_payload_total:,} bytes total "
        f"(Zerobus scales with payload bytes)"
    )
    if _singles:
        print(
            f"  single-row phase: {_bytes_4a:,} bytes (~{_bytes_4a / _singles:.0f} bytes/row avg)"
        )
    if _bytes_4b and _batch_n:
        print(
            f"  batch phase: {_bytes_4b:,} bytes (~{_bytes_4b / _batch_n:.1f} bytes/row avg)"
        )
print()
print()

print(
    f"After ingest: expected_count={_target_count} visible_count={_row_visible} "
    f"visibility_poll_s={_visibility_s:.3f} (time until COUNT(*) reached target)"
)
if _row_visible < _target_count:
    print("  Warning: COUNT still short after poll window; re-run or raise poll budget.")

_bounds_after = spark.sql(
    f"SELECT MIN(device_name) AS lo, MAX(device_name) AS hi FROM {TABLE_NAME}"
).collect()[0]
print(
    f"After visibility: min(device_name)={_bounds_after['lo']!r} max(device_name)={_bounds_after['hi']!r}"
)

# SDK ingest timing (4a singles / 4b batch); printed here with visibility for one recap block
print(f"Ingested {_n} rows into {TABLE_NAME}")
print(f"  Ingest+wait wall (4a+4b only; excludes stream.close): {_ingest_4a4b_s:.3f}s")
if _singles:
    _ms_row_4a = (_singles_wall_s / _singles) * 1000
    print(
        f"  4a ({_singles} singles): wall {_singles_wall_s:.3f}s  (~{_ms_row_4a:.1f} ms/row amortized)"
    )
    print(
        f"      per-row ingest+wait: min={min(_row_ack_seconds)*1000:.1f}ms "
        f"mean={mean(_row_ack_seconds)*1000:.1f}ms median={median(_row_ack_seconds)*1000:.1f}ms "
        f"max={max(_row_ack_seconds)*1000:.1f}ms"
    )
if _batch_ack_s is not None and _batch_n:
    _ms_row_4b = (_batch_ack_s / _batch_n) * 1000
    print(
        f"  4b ({_batch_n} batched): wall {_batch_ack_s:.3f}s  (~{_ms_row_4b:.3f} ms/row amortized)"
    )
if _singles and _batch_ack_s is not None and _batch_n:
    _ms_row_4a = (_singles_wall_s / _singles) * 1000
    _ms_row_4b = (_batch_ack_s / _batch_n) * 1000
    _ratio = _ms_row_4a / _ms_row_4b
    print(
        f"  Batch vs singles (wall amortized ms/row): 4a {_ms_row_4a:.1f} / 4b {_ms_row_4b:.3f} "
        f"→ ~{_ratio:.1f}x lower ms/row in 4b"
    )


### Optional: acknowledgment callback

From the same docs page: use `ack_callback` with a subclass of `AckCallback` (`on_ack` / `on_error`).

In [ ]:
from zerobus.sdk.shared import AckCallback, StreamConfigurationOptions, RecordType


class MyAckCallback(AckCallback):
    def on_ack(self, offset: int) -> None:
        print(f"Record acknowledged at offset: {offset}")

    def on_error(self, offset: int, error_message: str) -> None:
        print(f"Error at offset {offset}: {error_message}")


options_with_ack = StreamConfigurationOptions(
    record_type=RecordType.JSON,
    ack_callback=MyAckCallback(),
)

# Example: pass options_with_ack into create_stream(...) instead of `options` above.
print("Define options_with_ack; use in Step 4 setup cell when calling create_stream(...) instead of `options`.")


### Protocol Buffers

For type-safe ingestion, use **Protocol Buffers** with `RecordType.PROTO` (default) and provide a `descriptor_proto` in table properties. See the **[Python SDK repository](https://github.com/databricks/zerobus-sdk-py)** for `generate_proto`, batch ingestion, and full configuration options.

In [ ]:
# Sample rows (marketing: each row is often ack'd in ~250ms; full table typically visible within a few seconds)
display(spark.sql(f"SELECT * FROM {TABLE_NAME} ORDER BY device_name LIMIT 20"))


### Step 5: Zerobus system tables

Zerobus **ingest** and **stream** telemetry in `system.lakeflow` (Beta; see [Zerobus system tables](https://docs.databricks.com/aws/en/admin/system-tables/zerobus-ingest)). Rows can lag; tables must be enabled for your workspace.


#### 5.a. `zerobus_ingest`

All columns from `system.lakeflow.zerobus_ingest` for this table (`SELECT *`, filtered by `table_name` and `workspace_id`, `ORDER BY commit_time DESC`, `LIMIT 20`).


In [ ]:
# Step 5a: commit batches (zerobus_ingest)
# there will be some delays

try:
    display(spark.sql(f"""
    SELECT *
    FROM system.lakeflow.zerobus_ingest
    WHERE table_name = '{TABLE_NAME}'
    AND workspace_id = '{DATABRICKS_WORKSPACE_ID}'
    ORDER BY commit_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_ingest is not available in this workspace.")
        print("Ask a workspace admin to enable the Zerobus Ingest system table.")
    else:
        raise


#### 5.b. `zerobus_stream`

All columns from `system.lakeflow.zerobus_stream` for this table (same `table_name` / `workspace_id` filters, `LIMIT 20`).


In [ ]:
# Step 5b: stream events (zerobus_stream)

try:
    display(spark.sql(f"""
    SELECT *
    FROM system.lakeflow.zerobus_stream
    WHERE table_name = '{TABLE_NAME}'
    AND workspace_id = '{DATABRICKS_WORKSPACE_ID}'
    ORDER BY event_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_stream is not available in this workspace.")
        print("Ask a workspace admin to enable the Zerobus stream system table.")
    else:
        raise
